# Object-Oriented Programming for Data Engineering

This notebook covers Object-Oriented Programming concepts using Python with data engineering examples.

Topics covered:
- Classes and Objects
- `__init__` constructor
- Inheritance
- Encapsulation
- Polymorphism
- Abstraction
- Data pipeline style examples
- Interview questions


## 1. Why OOP is useful in Data Engineering

In data engineering, many components behave like reusable systems:

- database connectors
- file loaders
- API clients
- data validators
- pipeline processors
- loggers
- cloud storage clients

OOP helps represent these systems as reusable objects.

Example:

A `DatabaseConnector` class can act as a blueprint.

Each actual database connection can be created as an object.


## 2. Class and Object

A class is a blueprint.

An object is a real instance created from that blueprint.


In [ ]:
class DatabaseConnector:
    pass

db_connection = DatabaseConnector()

print(type(db_connection))


## 3. Simple Data Engineering Class


In [ ]:
class DataPipeline:
    def run(self):
        print("Pipeline started")
        print(type(self))

pipeline = DataPipeline()
pipeline.run()


## 4. The `__init__` Constructor

The `__init__` method runs automatically when an object is created.

It is used to initialize object attributes.


In [ ]:
class DatabaseConnector:
    def __init__(self, host, database):
        self.host = host
        self.database = database

    def connect(self):
        print(f"Connecting to database '{self.database}' on host '{self.host}'")

postgres_conn = DatabaseConnector("localhost", "sales_db")
postgres_conn.connect()


## 5. Object Attributes

Attributes store object-specific data.


In [ ]:
class FileMetadata:
    def __init__(self, file_name, file_type, row_count):
        self.file_name = file_name
        self.file_type = file_type
        self.row_count = row_count

file_1 = FileMetadata("orders.csv", "csv", 1000)
file_2 = FileMetadata("customers.json", "json", 500)

print(file_1.file_name, file_1.file_type, file_1.row_count)
print(file_2.file_name, file_2.file_type, file_2.row_count)


## 6. Object Methods

Methods define behavior for an object.


In [ ]:
class DataFile:
    def __init__(self, file_name, row_count):
        self.file_name = file_name
        self.row_count = row_count

    def describe(self):
        print(f"File: {self.file_name}")
        print(f"Rows: {self.row_count}")

    def is_large_file(self):
        return self.row_count > 100000

file_obj = DataFile("transactions.csv", 250000)
file_obj.describe()
print("Large file:", file_obj.is_large_file())


## 7. Example: API Client Class

A third-party API client can be represented as a class.


In [ ]:
class EcommerceAPIClient:
    def __init__(self, base_url):
        self.base_url = base_url

    def get_products_url(self):
        return self.base_url + "/api/products"

    def get_orders_url(self):
        return self.base_url + "/api/orders"

client = EcommerceAPIClient("https://fake-store-api.mock.beeceptor.com")

print(client.get_products_url())
print(client.get_orders_url())


# The Four Pillars of OOP


## 8. Pillar 1 — Inheritance

Inheritance allows a child class to reuse properties and methods from a parent class.

In data engineering, a generic loader can be created as a base class.

Specific loaders like `LocalLoader`, `S3Loader`, or `APILoader` can inherit from it.


In [ ]:
class FileLoader:
    def __init__(self, source_path):
        self.source_path = source_path

    def validate_path(self):
        print(f"Validating path: {self.source_path}")

class LocalFileLoader(FileLoader):
    def load(self):
        self.validate_path()
        print(f"Loading file from local path: {self.source_path}")

local_loader = LocalFileLoader("/data/orders.csv")
local_loader.load()


## 9. Inheritance with Multiple Child Classes


In [ ]:
class FileLoader:
    def __init__(self, source_path):
        self.source_path = source_path

    def validate_path(self):
        print(f"Checking source path: {self.source_path}")

class LocalLoader(FileLoader):
    def load(self):
        self.validate_path()
        print("Loading data from local file system")

class S3Loader(FileLoader):
    def load(self):
        self.validate_path()
        print("Loading data from Amazon S3")

class AzureBlobLoader(FileLoader):
    def load(self):
        self.validate_path()
        print("Loading data from Azure Blob Storage")

local = LocalLoader("/data/orders.csv")
s3 = S3Loader("s3://bucket/orders/")
azure = AzureBlobLoader("abfss://container/orders/")

local.load()
s3.load()
azure.load()


## 10. Why Inheritance is useful in data pipelines

Common behavior can be placed in the base class.

Specific behavior can be implemented in child classes.

Example:
- all loaders need path validation
- local loader reads local files
- S3 loader reads cloud storage
- API loader reads from an endpoint


## 11. Pillar 2 — Encapsulation

Encapsulation means restricting direct access to internal data.

In data engineering, this is useful for protecting sensitive details such as:
- API keys
- passwords
- database credentials
- tokens

Python naming conventions:
- `_variable` means protected/internal use
- `__variable` means private name-mangled attribute


In [ ]:
class APIClient:
    def __init__(self, base_url, api_key):
        self.base_url = base_url
        self._api_key = api_key

    def get_headers(self):
        return {
            "Authorization": f"Bearer {self._api_key}"
        }

client = APIClient("https://api.example.com", "secret-key-123")

print(client.base_url)

print(client._api_key)
print(client.get_headers())


## 12. Private Attributes with Double Underscore


In [ ]:
class SecureDatabaseConnector:
    def __init__(self, host, username, password):
        self.host = host
        self.username = username
        self.__password = password

    def connect(self):
        print(f"Connecting to {self.host} as {self.username}")
        print("Password is hidden inside the class")

db = SecureDatabaseConnector("warehouse.company.com", "etl_user", "super-secret-password")
db.connect()

# Direct access will fail:
print(db.__password)


## 13. Controlled Access Using Methods


In [ ]:
class SecureAPIClient:
    def __init__(self, api_key):
        self.__api_key = api_key

    def update_api_key(self, new_key):
        if len(new_key) >= 10:
            self.__api_key = new_key
            print("API key updated")
        else:
            print("Invalid API key")

    def masked_key(self):
        return self.__api_key[:3] + "****" + self.__api_key[-3:]

client = SecureAPIClient("abc123456789")
print(client.masked_key())

client.update_api_key("short")
client.update_api_key("newapikey98765")
print(client.masked_key())


## 14. Why Encapsulation is useful in data pipelines

A pipeline often needs secrets and configuration values.

Encapsulation prevents accidental changes and hides sensitive data from direct access.

Example:
- API keys should not be printed directly
- passwords should not be modified casually
- configuration updates should follow validation rules


## 15. Pillar 3 — Polymorphism

Polymorphism means the same method name can behave differently for different objects.

In data engineering, this is useful when different components follow the same interface.

Example:
- CSV processor has `process()`
- JSON processor has `process()`
- API processor has `process()`

The calling code can use `.process()` without worrying about the internal implementation.


In [ ]:
class CSVProcessor:
    def process(self):
        print("Reading CSV file")
        print("Cleaning tabular rows")
        print("Writing cleaned CSV output")

class JSONProcessor:
    def process(self):
        print("Reading JSON file")
        print("Flattening nested fields")
        print("Writing cleaned JSON output")

class APIProcessor:
    def process(self):
        print("Calling API endpoint")
        print("Validating API response")
        print("Writing API data output")

processors = [CSVProcessor(), JSONProcessor(), APIProcessor()]

for processor in processors:
    processor.process()
    print("---")


## 16. Polymorphism with File Loaders


In [ ]:
class LocalLoader:
    def load(self):
        return "Loaded data from local file"

class S3Loader:
    def load(self):
        return "Loaded data from S3"

class APILoader:
    def load(self):
        return "Loaded data from API"

loaders = [LocalLoader(), S3Loader(), APILoader()]

for loader in loaders:
    data = loader.load()
    print(data)


## 17. Why Polymorphism is useful in data pipelines

The pipeline can work with different sources using the same method name.

This makes the code flexible and easy to extend.

A new loader can be added without changing the pipeline logic.


## 18. Pillar 4 — Abstraction

Abstraction means defining a common template while hiding implementation details.

In Python, abstraction can be implemented using Abstract Base Classes.

An abstract class defines methods that child classes must implement.


In [ ]:
from abc import ABC, abstractmethod

class DataSource(ABC):
    @abstractmethod
    def extract(self):
        pass

class CSVSource(DataSource):
    def extract(self):
        print("Extracting data from CSV file")

class APISource(DataSource):
    def extract(self):
        print("Extracting data from API")

csv_source = CSVSource()
api_source = APISource()

csv_source.extract()
api_source.extract()


## 19. Abstraction with ETL Components


In [ ]:
from abc import ABC, abstractmethod

class PipelineStep(ABC):
    @abstractmethod
    def execute(self):
        pass

class ExtractStep(PipelineStep):
    def execute(self):
        print("Extracting raw data")

class TransformStep(PipelineStep):
    def execute(self):
        print("Transforming and cleaning data")

class LoadStep(PipelineStep):
    def execute(self):
        print("Loading data into target system")

steps = [ExtractStep(), TransformStep(), LoadStep()]

for step in steps:
    step.execute()


## 20. Why Abstraction is useful in data pipelines

Abstraction creates a standard structure.

Every pipeline step must follow the same contract.

This helps large teams build consistent components.

Example:
- every source must implement `extract()`
- every processor must implement `process()`
- every destination must implement `load()`


# Complete Data Engineering Example


## 21. Building a Small OOP-Based Data Pipeline

This example uses:
- inheritance
- encapsulation
- polymorphism
- abstraction


In [ ]:
from abc import ABC, abstractmethod

class Extractor(ABC):
    @abstractmethod
    def extract(self):
        pass

class MockAPIExtractor(Extractor):
    def __init__(self, source_name):
        self.source_name = source_name

    def extract(self):
        return [
            {"order_id": "1001", "amount": "250.50", "status": "completed"},
            {"order_id": "1002", "amount": "-50", "status": "completed"},
            {"order_id": "1003", "amount": "500.00", "status": "pending"},
            {"order_id": "1004", "amount": "1200.00", "status": "completed"}
        ]

class OrderTransformer:
    def transform(self, records):
        clean_records = []

        for record in records:
            order_id = int(record["order_id"])
            amount = float(record["amount"])
            status = record["status"]

            if amount > 0 and status == "completed":
                clean_records.append({
                    "order_id": order_id,
                    "amount": amount,
                    "tax": round(amount * 0.18, 2),
                    "final_amount": round(amount * 1.18, 2)
                })

        return clean_records

class ConsoleLoader:
    def load(self, records):
        for record in records:
            print(record)

extractor = MockAPIExtractor("Fake Store API")
transformer = OrderTransformer()
loader = ConsoleLoader()

raw_data = extractor.extract()
clean_data = transformer.transform(raw_data)
loader.load(clean_data)


## 22. Same Pipeline with a Pipeline Class


In [ ]:
class DataPipeline:
    def __init__(self, extractor, transformer, loader):
        self.extractor = extractor
        self.transformer = transformer
        self.loader = loader

    def run(self):
        raw_data = self.extractor.extract()
        clean_data = self.transformer.transform(raw_data)
        self.loader.load(clean_data)

pipeline = DataPipeline(
    extractor=MockAPIExtractor("Fake Store API"),
    transformer=OrderTransformer(),
    loader=ConsoleLoader()
)

pipeline.run()


## 23. Adding Another Loader Using Polymorphism


In [ ]:
class SummaryLoader:
    def load(self, records):
        print("Total clean records:", len(records))
        total_amount = 0

        for record in records:
            total_amount += record["final_amount"]

        print("Total final amount:", round(total_amount, 2))

summary_pipeline = DataPipeline(
    extractor=MockAPIExtractor("Fake Store API"),
    transformer=OrderTransformer(),
    loader=SummaryLoader()
)

summary_pipeline.run()


## 24. Encapsulation in Pipeline Configuration


In [ ]:
class PipelineConfig:
    def __init__(self, environment, api_key):
        self.environment = environment
        self.__api_key = api_key

    def get_environment(self):
        return self.environment

    def get_masked_api_key(self):
        return self.__api_key[:3] + "****" + self.__api_key[-3:]

config = PipelineConfig("development", "abc123456789")

print(config.get_environment())
print(config.get_masked_api_key())


# Practice Problems


## 25. Practice Problem 1 — Product Class

Create a `Product` class with:
- product_id
- name
- price
- stock

Add methods:
- `is_available()`
- `apply_discount(percent)`
- `display()`


In [ ]:
class Product:
    def __init__(self, product_id, name, price, stock):
        self.product_id = product_id
        self.name = name
        self.price = price
        self.stock = stock

    def is_available(self):
        return self.stock > 0

    def apply_discount(self, percent):
        self.price = self.price - (self.price * percent / 100)

    def display(self):
        print({
            "product_id": self.product_id,
            "name": self.name,
            "price": round(self.price, 2),
            "stock": self.stock,
            "available": self.is_available()
        })

product = Product(101, "Laptop", 80000, 10)
product.apply_discount(10)
product.display()


## 26. Practice Problem 2 — FileLoader Inheritance

Create:
- a base class `FileLoader`
- child class `CSVLoader`
- child class `JSONLoader`

Both child classes should implement `load()`.


In [ ]:
class FileLoader:
    def __init__(self, path):
        self.path = path

    def validate_path(self):
        return self.path is not None and len(self.path) > 0

class CSVLoader(FileLoader):
    def load(self):
        if self.validate_path():
            print(f"Loading CSV from {self.path}")
        else:
            print("Invalid CSV path")

class JSONLoader(FileLoader):
    def load(self):
        if self.validate_path():
            print(f"Loading JSON from {self.path}")
        else:
            print("Invalid JSON path")

csv_loader = CSVLoader("/data/orders.csv")
json_loader = JSONLoader("/data/products.json")

csv_loader.load()
json_loader.load()


## 27. Practice Problem 3 — Abstract Data Validator

Create an abstract class `Validator`.

Create child validators:
- `OrderValidator`
- `ProductValidator`

Each should implement `validate()`.


In [ ]:
from abc import ABC, abstractmethod

class Validator(ABC):
    @abstractmethod
    def validate(self, record):
        pass

class OrderValidator(Validator):
    def validate(self, record):
        return record["amount"] > 0 and record["status"] == "completed"

class ProductValidator(Validator):
    def validate(self, record):
        return record["price"] > 0 and record["stock"] >= 0

order_validator = OrderValidator()
product_validator = ProductValidator()

order = {"amount": 250, "status": "completed"}
product = {"price": 999, "stock": 12}

print(order_validator.validate(order))
print(product_validator.validate(product))


# Interview Questions


## 28. Why is OOP important for building data pipelines?

OOP is important in data pipelines because it helps organize code into reusable components.

Examples:
- Extractors can handle different data sources.
- Transformers can clean and validate data.
- Loaders can write data to different destinations.
- Pipeline classes can combine these pieces into a reusable workflow.

This makes the code easier to maintain, test, extend, and reuse.


## 29. Difference between Inheritance and Abstraction

Inheritance:
- A child class reuses code from a parent class.
- Example: `S3Loader` inherits common path validation from `FileLoader`.

Abstraction:
- A base class defines what methods must exist, but not necessarily how they work.
- Example: `DataSource` requires every child class to implement `extract()`.

Inheritance focuses on code reuse.

Abstraction focuses on enforcing structure.


## 30. Difference between Encapsulation and Abstraction

Encapsulation:
- Hides internal data.
- Protects sensitive information.
- Example: hiding API keys or passwords.

Abstraction:
- Hides implementation details.
- Exposes a standard interface.
- Example: every data source has an `extract()` method, but the internal extraction logic may differ.


## 31. Difference between Polymorphism and Inheritance

Inheritance:
- Child classes receive behavior from parent classes.

Polymorphism:
- Different classes use the same method name but implement different behavior.

Example:
- `CSVProcessor.process()`
- `JSONProcessor.process()`
- `APIProcessor.process()`

All have the same method name, but the logic is different.


## 32. Summary

OOP in data engineering helps build reusable and scalable pipeline components.

Main ideas:
- Class: blueprint
- Object: actual instance
- Constructor: initializes attributes
- Inheritance: reuse common code
- Encapsulation: protect internal data
- Polymorphism: same method name, different behavior
- Abstraction: enforce a common structure
